# Neural Network Month · Project 1
## Notebook 1 of 3 — Building the Cohort

Welcome to November. This month we open the black box. Before we build a network we need something for it to learn, so today we create a **synthetic cohort of learners** and describe each one with three inputs and two outcomes:

| Inputs (what we measure) | Outcomes (what we predict) |
|---|---|
| Hours of sleep | Memory recall score (0-100) |
| Hours of study | Reaction time (milliseconds) |
| Stress level (1-10) | |

Predicting **two outcomes at once** is the point of the project. A classifier gives one label. A network with two output neurons has to learn a shared internal picture that serves both outcomes, which is closer to how a brain region serves more than one purpose.

**This notebook:** write the hidden rule that generates the data, create a small seed cohort, expand it with a *smoothed bootstrap*, and save everything for Notebooks 2 and 3. Three figures.

## 1 · Setup and palette

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path

# One palette for every figure in the series
NEURO_PALETTE = {
    "input":    "#7A9E7E",  # sage   - input neurons and features
    "hidden":   "#C97B4A",  # clay   - hidden neurons
    "recall":   "#2F6F73",  # teal   - recall target
    "reaction": "#D9A441",  # ochre  - reaction-time target
    "ink":      "#4A3F35",  # bark   - text, seed data, baselines
    "paper":    "#F5EFE6",  # sand   - background
    "grid":     "#D8CFC0",  # stone  - gridlines
}
P = NEURO_PALETTE
NEURO_CMAP = LinearSegmentedColormap.from_list("neuro", [P["recall"], P["paper"], P["hidden"]])

plt.rcParams.update({
    "figure.facecolor": P["paper"], "axes.facecolor": P["paper"],
    "axes.edgecolor": P["ink"], "axes.labelcolor": P["ink"],
    "text.color": P["ink"], "xtick.color": P["ink"], "ytick.color": P["ink"],
    "axes.grid": True, "grid.color": P["grid"], "grid.alpha": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "figure.titlesize": 16, "figure.titleweight": "bold",
    "font.size": 11, "figure.dpi": 110,
})

Path("figures").mkdir(exist_ok=True)
def save_fig(fig, name):
    fig.savefig(f"figures/{name}.png", dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())

## 2 · The hidden rule

Real learners are messy, but a few relationships are well established: more sleep helps recall and speeds reactions, study helps with diminishing returns, and stress hurts both. The function below encodes those in simple form, plus random noise.

Two details matter for what comes later:

- **Curves, not lines.** Sleep and study gains flatten out (diminishing returns) and stress costs accelerate.
- **Two interactions.** Study only pays off fully when you are rested, and stress slows reactions more when you are sleep-deprived. A straight-line model cannot express either cleanly, which is where a hidden layer earns its keep.

This is invented data. It illustrates a mechanism; it is not a claim about any real population.

In [ ]:
FEATURES = ["sleep_hrs", "study_hrs", "stress"]
TARGETS  = ["recall_score", "reaction_ms"]

def true_process(sleep, study, stress, rng=None):
    """The hidden 'biology' our network must rediscover. Noise only if rng is given."""
    sleep_norm  = np.clip((sleep - 4) / 4, 0, 1)
    sleep_gain  = 25 * (1 - np.exp(-(sleep - 4) / 2))                     # saturating
    study_gain  = 22 * (1 - np.exp(-study / 2)) * (0.5 + 0.5 * sleep_norm) # diminishing + needs sleep
    stress_cost = 0.35 * stress ** 1.5                                     # accelerating
    recall   = 35 + sleep_gain + study_gain - stress_cost
    reaction = 250 + 90 * np.exp(-(sleep - 4) / 2.5) * (1 + 0.1 * (stress - 1)) + 3 * (stress - 1) + 4 * np.maximum(study - 4, 0)
    if rng is not None:
        recall   = recall   + rng.normal(0, 3,  size=np.shape(recall))
        reaction = reaction + rng.normal(0, 8,  size=np.shape(reaction))
    return np.clip(recall, 0, 100), reaction

def make_seed(n, rng):
    """A small 'real-looking' cohort of learners."""
    sleep  = np.clip(rng.normal(6.8, 1.1, n), 4, 9.5)
    study  = rng.uniform(0.5, 6, n)
    stress = np.clip(rng.normal(5, 2, n), 1, 10)
    recall, reaction = true_process(sleep, study, stress, rng)
    return pd.DataFrame({"seed_id": np.arange(n), "sleep_hrs": sleep, "study_hrs": study,
                         "stress": stress, "recall_score": recall, "reaction_ms": reaction})

def smooth_bootstrap(seed_df, n_total, rng, jitter=0.05):
    """Resample whole rows with replacement, then nudge each value by 5% of its spread."""
    cols = FEATURES + TARGETS
    idx = rng.integers(0, len(seed_df), n_total)
    boot = seed_df.iloc[idx].reset_index(drop=True).copy()
    sd = seed_df[cols].std().values
    boot[cols] = boot[cols].values + rng.normal(0, 1, (n_total, len(cols))) * sd * jitter
    boot["sleep_hrs"]    = boot["sleep_hrs"].clip(4, 9.5)
    boot["study_hrs"]    = boot["study_hrs"].clip(0.5, 6)
    boot["stress"]       = boot["stress"].clip(1, 10)
    boot["recall_score"] = boot["recall_score"].clip(0, 100)
    return boot

## 3 · Seed cohort, then bootstrap

We start with **300 seed learners**, then use a **smoothed bootstrap** to grow to **1,500 rows**: resample whole rows with replacement, then nudge every value by 5% of its spread so the copies are not identical.

One honest caveat that shapes the rest of the project: bootstrapping adds **no new information**. Every row is still a relative of one of the 300 seeds. That has a practical consequence. When we validate later, we must keep all copies of a seed learner on the *same side* of the train/test line, or the network gets graded on people it has effectively already seen. Each row therefore carries its `seed_id`.

In [ ]:
rng = np.random.default_rng(42)
seed   = make_seed(300, rng)
cohort = smooth_bootstrap(seed, 1500, rng)

seed.to_csv("neuro_seed.csv", index=False)
cohort.to_csv("neuro_cohort.csv", index=False)
print(f"Seed learners: {len(seed)} | Bootstrapped rows: {len(cohort)}")
cohort.describe().round(1)

## 4 · Figure 1 — what the cohort looks like

In [ ]:
labels = {"sleep_hrs": "Sleep Hours", "study_hrs": "Study Hours", "stress": "Stress Level",
          "recall_score": "Recall Score", "reaction_ms": "Reaction Time (ms)"}
colors = [P["input"]] * 3 + [P["recall"], P["reaction"]]

fig, axes = plt.subplots(1, 5, figsize=(17, 3.6))
for ax, col, c in zip(axes, FEATURES + TARGETS, colors):
    ax.hist(cohort[col], bins=22, color=c, edgecolor=P["paper"])
    ax.set_title(labels[col].split(" (")[0])
    ax.set_xlabel(labels[col]); ax.set_yticks([])
axes[0].set_ylabel("Learners")
fig.suptitle("Synthetic Cohort Distributions", y=1.05)
fig.tight_layout(); save_fig(fig, "01_cohort_distributions"); plt.show()

Green is an **input** and teal/ochre are the two **outcomes**. This colour logic holds for every figure in the series: sage = inputs, clay = hidden neurons, teal = recall, ochre = reaction time.

## 5 · Figure 2 — does the bootstrap keep the shape?

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.4))

a1.scatter(cohort["sleep_hrs"], cohort["recall_score"], s=10, alpha=0.35, color=P["input"], label="Bootstrapped (1,500)")
a1.scatter(seed["sleep_hrs"], seed["recall_score"], s=26, color=P["ink"], label="Seed (300)")
a1.set_xlabel("Sleep hours"); a1.set_ylabel("Recall score"); a1.set_title("Sleep Versus Recall")
a1.legend(frameon=False)

bins = np.linspace(cohort["recall_score"].min(), cohort["recall_score"].max(), 25)
a2.hist(cohort["recall_score"], bins=bins, density=True, color=P["input"], alpha=0.6, label="Bootstrapped")
a2.hist(seed["recall_score"], bins=bins, density=True, histtype="step", linewidth=2, color=P["ink"], label="Seed")
a2.set_xlabel("Recall score"); a2.set_yticks([]); a2.set_title("Recall Distribution")
a2.legend(frameon=False)

fig.suptitle("Seed Versus Bootstrapped Cohort", y=1.02)
fig.tight_layout(); save_fig(fig, "02_seed_vs_bootstrap"); plt.show()

The bootstrapped cloud wraps tightly around the seed points and keeps the same distribution, which is what a smoothed bootstrap should do.

## 6 · Figure 3 — sleep touches both outcomes

In [ ]:
def binned_mean(x, y, n_bins=12):
    edges = np.linspace(x.min(), x.max(), n_bins + 1)
    mid = (edges[:-1] + edges[1:]) / 2
    idx = np.clip(np.digitize(x, edges) - 1, 0, n_bins - 1)
    return mid, np.array([y[idx == i].mean() for i in range(n_bins)])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.4))
for ax, col, c, title, yl in [(a1, "recall_score", P["recall"], "Sleep and Recall", "Recall score"),
                              (a2, "reaction_ms", P["reaction"], "Sleep and Reaction", "Reaction time (ms)")]:
    ax.scatter(cohort["sleep_hrs"], cohort[col], s=10, alpha=0.3, color=c)
    mx, my = binned_mean(cohort["sleep_hrs"].values, cohort[col].values)
    ax.plot(mx, my, color=P["ink"], linewidth=2.5, label="Average trend")
    ax.set_xlabel("Sleep hours"); ax.set_ylabel(yl); ax.set_title(title); ax.legend(frameon=False)
fig.suptitle("Sleep Shapes Both Outcomes", y=1.02)
fig.tight_layout(); save_fig(fig, "03_sleep_both_outcomes"); plt.show()

More sleep raises recall and lowers reaction time, and both trends flatten out. Because the two outcomes share causes, a network with a **shared hidden layer** has a real reason to learn them together.

**Next:** Notebook 2 builds the network from scratch in NumPy and shows how backpropagation trains it.

In [ ]:
# Optional: download this notebook's figures
try:
    import shutil
    from google.colab import files
    shutil.make_archive("nb1_figures", "zip", "figures")
    files.download("nb1_figures.zip")
except ImportError:
    print("Figures are saved in the 'figures' folder.")